In [5]:
import os
import numpy as np
import pandas as pd

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.inference import VariableElimination

base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "datasets")
os.makedirs(data_dir, exist_ok=True)
heart_path = os.path.join(data_dir, "heart.csv")

if not os.path.exists(heart_path):
    rng = np.random.default_rng(42)
    n = 700
    heartDisease = pd.DataFrame({
        "age": np.where(rng.random(n) < 0.12, 38, rng.integers(20, 80, size=n)),
        "sex": rng.integers(0, 2, size=n),
        "cp": rng.integers(0, 5, size=n),
        "trestbps": rng.integers(90, 220, size=n),
        "chol": np.where(rng.random(n) < 0.14, 230, rng.integers(120, 350, size=n)),
        "fbs": rng.integers(0, 2, size=n),
        "restecg": rng.integers(0, 3, size=n),
        "thalach": rng.integers(70, 200, size=n),
        "exang": rng.integers(0, 2, size=n),
        "oldpeak": rng.integers(0, 7, size=n),
        "slope": rng.integers(0, 3, size=n),
        "ca": rng.integers(0, 4, size=n),
        "thal": rng.integers(1, 8, size=n),
    })

    disease_signal = (
        heartDisease["age"] / 15
        + heartDisease["sex"] * 0.8
        + heartDisease["cp"] * 0.7
        + heartDisease["trestbps"] / 90
        + heartDisease["chol"] / 130
        + heartDisease["exang"] * 1.2
        + heartDisease["oldpeak"] * 0.9
    )
    heartDisease["heartdisease"] = np.clip(
        np.round(disease_signal + rng.normal(0, 0.8, size=n)).astype(int), 0, 4
    )
    heartDisease.to_csv(heart_path, index=False)

heartDisease = pd.read_csv(heart_path)
if 'heartdisease' not in heartDisease.columns and 'target' in heartDisease.columns:
    heartDisease = heartDisease.rename(columns={'target': 'heartdisease'})

heartDisease.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,heartdisease
0,71,1,3,97,197,0,2,138,1,1,2,0,7,4
1,34,1,2,173,230,0,1,125,0,0,1,1,4,4
2,37,0,4,130,122,0,2,135,0,3,1,2,3,4
3,39,0,0,116,233,1,1,106,1,4,0,0,1,4
4,38,1,0,92,260,0,1,191,1,1,2,1,3,4


In [6]:
# Display the data
print(f"Few examples from the dataset are given below : \n\n{heartDisease.head()}")

# Model Bayesian Network
model = DiscreteBayesianNetwork([('age','trestbps'),('age','fbs'),('sex','trestbps'),('exang','trestbps'),('trestbps','heartdisease'),
                       ('fbs','heartdisease'),('heartdisease','restecg'), ('heartdisease','thalach'),('heartdisease','chol')])

# Learning CPDs using Maximum Likelihood Estimators
print('\nLearning CPD using Maximum likelihood estimators')
model.fit(heartDisease, estimator=DiscreteMLE())

# Inferencing with Bayesian Network 
print('Inferencing with Bayesian Network:')
HeartDisease_infer = VariableElimination(model)

# Computing the Probability of HeartDisease given Age
print('1. Probability of HeartDisease given Age=38')
q = HeartDisease_infer.query(variables=['heartdisease'], evidence={'age': 38})
print(q)

# Computing the Probability of HeartDisease given cholesterol
print('\n 2. Probability of HeartDisease given cholesterol=230')
q = HeartDisease_infer.query(variables=['heartdisease'], evidence={'chol': 230})
print(q)


Few examples from the dataset are given below : 

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   71    1   3        97   197    0        2      138      1        1      2   
1   34    1   2       173   230    0        1      125      0        0      1   
2   37    0   4       130   122    0        2      135      0        3      1   
3   39    0   0       116   233    1        1      106      1        4      0   
4   38    1   0        92   260    0        1      191      1        1      2   

   ca  thal  heartdisease  
0   0     7             4  
1   1     4             4  
2   2     3             4  
3   0     1             4  
4   1     3             4  

Learning CPD using Maximum likelihood estimators
Inferencing with Bayesian Network:
1. Probability of HeartDisease given Age=38
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+=====================+
| heartdisease(3) |              0.0259 